# 🔧 Notebook 2 — Data Preprocessing

**Course:** PA2595 Machine Learning Engineering

---

## What is preprocessing?

Raw data is almost never ready to be fed directly into a machine learning model.  
**Preprocessing** is the step where we:

1. **Create the target variable** — define what we want to predict (Pass / Fail)
2. **Encode categorical columns** — convert text values like "yes"/"no" into numbers
3. **Select features** — choose which columns to include in the model
4. **Handle missing values** — fill or remove any gaps in the data
5. **Split the dataset** — separate into training set (teaches the model) and test set (evaluates it)

> ⚠️ Always run this notebook **after** `01_data_exploration.ipynb`.

## Step 1 — Import Libraries

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))  # allow importing from src/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## Step 2 — Load the Raw Dataset

In [ ]:
df = pd.read_csv("../data/raw/student-mat.csv", sep=";")
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(3)

## Step 3 — Create the Target Variable

We define a new column called `target`:
- **1 (Pass)** → final grade G3 ≥ 10
- **0 (Fail)** → final grade G3 < 10

This is the value our model will learn to predict.

In [ ]:
PASS_THRESHOLD = 10

df["target"] = (df["G3"] >= PASS_THRESHOLD).astype(int)

print("Target value counts:")
print(df["target"].value_counts().rename({1: "Pass (1)", 0: "Fail (0)"}))

## Step 4 — Encode Categorical Columns

Machine learning algorithms work with **numbers only**.  
Columns like `internet = "yes"/"no"` must be converted to `1 / 0`.

We use `LabelEncoder` from scikit-learn, which assigns a unique integer to each category.

In [ ]:
# Columns that contain text categories and need to be encoded
BINARY_COLS = [
    "internet", "higher", "sex", "address", "famsize", "Pstatus",
    "schoolsup", "famsup", "paid", "activities", "nursery", "romantic",
]

le = LabelEncoder()

for col in BINARY_COLS:
    original_values = df[col].unique()
    df[col] = le.fit_transform(df[col])
    encoded_values = df[col].unique()
    print(f"  {col}: {sorted(original_values)} -> {sorted(encoded_values)}")

print("\nEncoding complete.")

## Step 5 — Select Features

Not all 33 columns are useful for prediction. We select only features that:
- Are related to student behaviour, background, or academic history
- Do **not** include G3 itself (that is the answer — using it would be cheating)

We keep G1 and G2 (interim grades) because they are strong predictors available before the final exam.

In [ ]:
SELECTED_FEATURES = [
    "studytime", "absences", "failures", "G1", "G2",
    "Medu", "Fedu", "traveltime", "freetime", "goout",
    "Dalc", "Walc", "health", "internet", "higher",
    "sex", "address", "famsize", "Pstatus",
    "schoolsup", "famsup", "paid", "activities", "nursery", "romantic",
]

# Only keep features that actually exist in the dataset
available = [f for f in SELECTED_FEATURES if f in df.columns]
print(f"Features selected: {len(available)}")
print(available)

## Step 6 — Handle Missing Values

We fill any numeric gaps with the **median** of that column.

The **median** (middle value when sorted) is more robust than the mean when there are outliers.

In [ ]:
X = df[available].copy()
y = df["target"].copy()

before = X.isnull().sum().sum()
X = X.fillna(X.median(numeric_only=True))
after = X.isnull().sum().sum()

print(f"Missing values before: {before} -> after: {after}")
print(f"Shape of feature matrix: {X.shape}")

## Step 7 — Split into Training and Test Sets

We split the data into:
- **Training set (80%)** — the model learns from this data
- **Test set (20%)** — we use this to measure how well the model generalises

**Why keep a separate test set?**  
If we evaluate the model on the same data it was trained on, the result would be misleadingly good.

We use `stratify=y` to ensure both sets have the same ratio of Pass/Fail.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% goes to the test set
    random_state=42,     # fixed seed for reproducibility
    stratify=y           # keeps Pass/Fail ratio the same in both sets
)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"\nTraining target distribution:\n{y_train.value_counts().rename({1:'Pass',0:'Fail'})}")
print(f"\nTest target distribution:\n{y_test.value_counts().rename({1:'Pass',0:'Fail'})}")

## Step 8 — Visualise the Split

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (data, title) in zip(axes, [
    (y_train, "Training Set"), (y_test, "Test Set")
]):
    counts = data.value_counts().rename({1: "Pass", 0: "Fail"})
    counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"], edgecolor="black")
    ax.set_title(f"{title} - Pass vs Fail", fontsize=13)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                str(int(bar.get_height())),
                ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## Step 9 — Save Processed Data

We save the four resulting CSV files to `data/processed/`.  
These will be loaded directly by the training notebook.

In [ ]:
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print("Saved files:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f} ({size:,} bytes)")

## ✅ Summary

| Step | What we did |
|---|---|
| Target creation | G3 ≥ 10 → Pass (1), else Fail (0) |
| Encoding | 12 categorical columns converted from text to 0/1 |
| Feature selection | 25 features selected |
| Missing values | Filled with column median |
| Train/test split | 80% training / 20% test, stratified |
| Output | 4 CSV files saved in `data/processed/` |

> 📌 **Next step:** Open notebook `03_model_training.ipynb` to train and compare machine learning models.